# 6 · Saving and loading: snapshots

**The problem.** The address book has to be saved to disk, sent to a browser, and loaded back, possibly by a program
in another language. The usual approach is `JSON.stringify` on nested objects, plus a hand-written reviver per type.
That breaks down on exactly the data from the last case studies:

- A shared address nested under two contacts is written twice and loads back as **two** addresses.
- A cycle (Byron mentors Ada, who is his mother) makes `JSON.stringify` throw.
- A link filled by "a Person *or* an Institution" loses its kind unless you invent a type tag.

The framework serializes through one intermediate form, **plain data**: maps, arrays, strings, numbers, booleans and
`null`. JSON and YAML (case study 7) are thin text encodings of it. The shape is a *snapshot*: a flat table of
objects keyed by *symbols*, with links written as references to symbols. Nothing is nested, so nothing is duplicated
and cycles are harmless.

In [1]:
import { Errors, JSON as Json, Plain, Proxies, Schemas, Validators } from "../src/Framework/index.js";
import { entries } from "./toolkit.js";

const { DecodeError } = Errors;

let Contact = new Schemas.OfObject.Builder()
  .properties(
    (p) => p.name("name").of((t) => t.as_native(String)),
    (p) => p.name("photo").of((t) => t.as_native(Uint8Array)),
    (p) => p.name("rating").of((t) => t.as_native(Number)),
  )
  .create();
let Address = new Schemas.OfObject.Builder().properties((p) => p.name("street").of((t) => t.as_native(String))).create();
const ContactAddresses = new Schemas.OfRelation.Builder()
  .links("contact", "address")
  .properties((p) => p.name("label").of((t) => t.as_native(String)))
  .create();
Contact = new Schemas.OfObject.Builder(Contact).relations((adj) => adj.name("addresses").of(ContactAddresses).me("contact")).update();
Address = new Schemas.OfObject.Builder(Address).relations((adj) => adj.name("residents").of(ContactAddresses).me("address")).update();
Proxies.register("Contact", Contact);
Proxies.register("Address", Address);
Proxies.register("ContactAddresses", ContactAddresses);
const Builders = Proxies.Builders;

const home = Builders.Address().street("7 Old Road").create();
const alice = Builders.Contact()
  .name("Alice")
  .photo(new Uint8Array([0x89, 0x50, 0x4e, 0x47, 0x0d, 0x0a]))
  .rating(Infinity)
  .addresses((x: any) => x.address(home).label("home"))
  .create();
const reggie = Builders.Contact().name("Reginald").addresses((x: any) => x.address(home).label("home")).create();

## One object

`Plain.ToPlain(schema, value)` converts a value, given its schema. In TypeScript, plain mappings are `Map`s rather
than object literals: a `Map` keeps every key in insertion order (object literals move integer-like keys such as
`"2"` to the front) and treats `"__proto__"` as an ordinary key. To read the result comfortably, print it as JSON
with the framework's own `JSON.dumps`, imported as `Json` so it doesn't shadow JavaScript's global `JSON`:

In [2]:
const one = Plain.ToPlain(Contact, alice);
console.log(Json.dumps(one, { indent: 2 }));

{
  "root": "s0",
  "objects": {
    "s0": {
      "name": "Alice",
      "photo": "iVBORw0K",
      "rating": "Infinity",
      "addresses": [
        {
          "address": {
            "$ref": "s1",
            "$schema": "Address"
          },
          "label": "home"
        }
      ]
    }
  }
}


Read it carefully. Every part is a decision:

- **`root` and `objects`.** Objects live in a flat table under *symbols* (`s0`, `s1`, ...), assigned in the order
  objects are first referenced. `root` says which one you asked for.
- **No schema in the object.** `s0` doesn't say it's a `Contact`. You passed the schema in, and you'll pass it in
  again when loading. The data stays small, and the meaning comes from the schema, not from tags scattered through
  the data.
- **Native values in plain form.** The `Uint8Array` became base64 text and `Infinity` became the string
  `"Infinity"`, so the data fits in strict JSON. The conversion is done by the native type's schema, once, not by
  each text format.
- **Entries under the adjacency.** The entry lists the *other* links and the properties. The object's own link is
  implied by where the entry appears.
- **References carry the schema name.** `{"$ref": "s1", "$schema": "Address"}` points at an object by symbol and
  says what it is. Links are untyped (case study 2), and a link may be filled by different kinds (case study 4), so
  the reference itself must say. This is why every linked object's schema must be registered under a name.

But `s1` isn't in `objects`. This snapshot covers *one* object, so its references dangle. That's fine for looking at
one record, but it can't be loaded on its own:

In [3]:
try {
  Plain.FromPlain(Proxies.Builders)(Contact, one);
} catch (error) {
  console.log(String(error));
}

DecodeError: $.objects.s0.addresses[0].address: unresolved reference 's1': the snapshot does not contain it


Loading refuses rather than inventing an empty address or dropping the entry. Either would silently change the data.
The error says *where* in the snapshot the problem is, as a path from the top (`$`); more on these errors below.

## A whole graph

`Plain.ToPlain.Reachable` saves the root and everything reachable from it: the connected component from case study
4. Every reference now resolves:

In [4]:
const graph = Plain.ToPlain.Reachable(Contact, alice);
console.log(Json.dumps(graph, { indent: 2 }));

{
  "root": "s0",
  "objects": {
    "s0": {
      "name": "Alice",
      "photo": "iVBORw0K",
      "rating": "Infinity",
      "addresses": [
        {
          "address": {
            "$ref": "s1",
            "$schema": "Address"
          },
          "label": "home"
        }
      ]
    },
    "s1": {
      "street": "7 Old Road",
      "residents": [
        {
          "contact": {
            "$ref": "s0",
            "$schema": "Contact"
          },
          "label": "home"
        },
        {
          "contact": {
            "$ref": "s2",
            "$schema": "Contact"
          },
          "label": "home"
        }
      ]
    },
    "s2": {
      "name": "Reginald",
      "addresses": [
        {
          "address": {
            "$ref": "s1",
            "$schema": "Address"
          },
          "label": "home"
        }
      ]
    }
  }
}


The shared address appears **once** (`s1`), and both contacts refer to it. Each entry appears under *every* object
it links, from that object's point of view: under Alice as `address: s1`, and under the address as `contact: s0`.
That makes each object's record complete on its own. When loading, the duplicate views are recognized as one fact.

## Loading

`Plain.FromPlain(builders)` is constructed with the **builders** to build with, then called like `ToPlain`:

In [5]:
const load = Plain.FromPlain(Proxies.Builders);
const alice2 = load.Reachable(Contact, graph) as Proxies.Instance;

console.log(alice2 === alice, alice2.name, alice2.photo, alice2.rating);
console.log(entries(alice2, "addresses").map((e) => [e.address.street, e.label]));
console.log(entries(entries(alice2, "addresses")[0]?.address, "residents").map((e) => [e.contact.name, e.label]));

false Alice Uint8Array(6) [ 137, 80, 78, 71, 13, 10 ] Infinity


[ [ "7 Old Road", "home" ] ]


[ [ "Alice", "home" ], [ "Reginald", "home" ] ]


A new, separate graph came back: new objects, the same structure, each entry once, and the shared address shared.

### Why pass the builders in?

The loader doesn't create objects itself. It asks the registry you give it for a builder by schema name, and drives
that builder like any other code would. Today the registry is `Proxies.Builders`. When generated bindings exist,
you'll pass theirs and get typed classes back from the *same* snapshot. In tests you can pass a registry that
records or refuses. The serializer never needs to know which implementation is in use.

## Serializers check types

In case studies 1 and 5 the builder accepted a string where bytes belonged, and the validator reported it.
Serializers are the second checkpoint. They refuse to write data that doesn't match its schema, so a bad value can't
reach disk and come back as something else. Writing a wrong type is a mistake in *your program*, so it's an ordinary
`TypeError`:

In [6]:
const sloppy = Builders.Contact().name("Bob").photo("not bytes").create();
try {
  Plain.ToPlain(Contact, sloppy);
} catch (error) {
  console.log(String(error));
}

TypeError: expected bytes, got str


## Loading errors: `DecodeError`

Loading is just as strict. Types are never coerced: an integer `1n` doesn't become the float `1`, and `"1"` doesn't
become `1`. But a wrong type in *loaded* data isn't a mistake in your program. It's a problem in the input, which
your program must be ready for. So every problem found while loading throws one class, `Errors.DecodeError` (a
`ValueError`), whichever layer finds it:

In [7]:
const snapshotOf = (fields: Record<string, unknown>) =>
  new Map<string, unknown>([["root", "s0"], ["objects", new Map([["s0", new Map(Object.entries(fields))]])]]);

for (const bad of [{ rating: 1n }, { rating: "fast" }, { photo: "not base64!" }, { name: null }]) {
  try {
    load(Contact, snapshotOf(bad));
  } catch (error) {
    if (!(error instanceof DecodeError)) throw error;
    console.log(`${Object.keys(bad)[0]?.padEnd(8)} -> ${error.message}`);
  }
}

rating   -> $.objects.s0.rating: expected float, got int


rating   -> $.objects.s0.rating: expected a float or one of ['-Infinity', 'Infinity', 'NaN'], got 'fast'


photo    -> $.objects.s0.photo: invalid base64 text


name     -> $.objects.s0.name: expected str, got NoneType


Each error has three parts you can use in code: `reason` (one line, no location), and either `path` (where in plain
data) or `line` and `column` (where in text, as case study 7 shows):

In [8]:
try {
  load(Contact, snapshotOf({ rating: 1n }));
} catch (error) {
  const e = error as InstanceType<typeof DecodeError>;
  console.log(e.path, "|", e.reason, "|", e.line, e.column);
}

$.objects.s0.rating | expected float, got int | null null


**Why one class?** A program loading a file wants to say "this file is bad, here's why and where" and carry on. With
errors from each underlying library (a JSON parser's `SyntaxError`, a YAML parser's own class, a base64 decoder's),
it would have to know every library the framework happens to use. With `DecodeError` it catches one class, and the
message reads the same in every language binding.

## Malformed snapshots are rejected, with a location

Snapshots come from outside your program, so their shape, references and values are all checked before anything is
built. If the check fails, nothing is built. The helper `at` walks into the plain data (maps by key, arrays by
index) to make each change:

In [9]:
// eslint-disable-next-line @typescript-eslint/no-explicit-any
const at = (value: any, ...keys: (string | number)[]): any => keys.reduce((v, k) => (typeof k === "number" ? v[k] : v.get(k)), value);
const mutated = (change: (s: unknown) => void) => {
  const snapshot = Json.loads(Json.dumps(graph)); // a deep copy, through text
  change(snapshot);
  return snapshot;
};

for (const [why, snapshot] of [
  ["unknown property", mutated((s) => at(s, "objects", "s0").set("colour", "red"))],
  ["missing object", mutated((s) => at(s, "objects").delete("s1"))],
  ["orphan object", mutated((s) => at(s, "objects").set("s9", new Map([["street", "?"]])))],
  ["wrong schema name", mutated((s) => at(s, "objects", "s0", "addresses", 0, "address").set("$schema", "Contact"))],
] as const) {
  try {
    load.Reachable(Contact, snapshot);
  } catch (error) {
    console.log(`${why.padEnd(18)} -> ${String((error as Error).message)}`);
  }
}

unknown property   -> $.objects.s0.colour: 'Contact' has no property or adjacency 'colour'


missing object     -> $.objects.s0.addresses[0].address: unresolved reference 's1': the snapshot does not contain it


orphan object      -> $.objects.s9: nothing references this object, so its schema is unknown


wrong schema name  -> $.objects.s2.addresses[0].address: 's1' is referenced as both 'Contact' and 'Address'


## Round trips

Saving what you loaded gives the same snapshot. Symbols may be *renumbered* when entries come back in a different
order, so compare structure, not symbol names, when order matters to you. (Whether snapshots should be fully
canonical is an open design question.)

In [10]:
const again = Plain.ToPlain.Reachable(Contact, alice2);
console.log(Json.dumps(again) === Json.dumps(graph));
console.log(Validators.Validate(Proxies.Builders).Reachable(Contact, alice2));

true


[]


## Any schema, not just objects

`ToPlain` and `FromPlain` dispatch on the kind of schema you pass. They also have per-kind entry points
(`.OfNative`, `.OfObject`, `.Reachable`) when you want to be explicit:

In [11]:
const Bytes = new Schemas.OfNative.Data(Uint8Array);
const two = new Uint8Array([1, 2]);
console.log(Plain.ToPlain(Bytes, two), Plain.ToPlain.OfNative(Bytes, two), load.OfNative(Bytes, "AQI="));

AQI= AQI= Uint8Array(2) [ 1, 2 ]


## What you learned

- A snapshot is `{root: symbol, objects: {symbol: object}}`, held as `Map`s. Objects are flat, and links are
  `{$ref: symbol, $schema: name}`.
- The schema is passed in, not stored. References carry schema names because links are untyped.
- `ToPlain` saves one object (references dangle). `ToPlain.Reachable` saves a whole component.
- `FromPlain(builders)` builds with the builders you give it. It refuses dangling references, malformed shapes and
  wrong types with `DecodeError`, which carries a path, and it builds nothing when it refuses.
- Natives have plain forms: base64 for bytes, and `"NaN"`/`"Infinity"`/`"-Infinity"` for non-finite floats.

**Next:** case study 7 turns snapshots into JSON and YAML text, where the exactness above keeps paying off.